[← All benchmarks](Evaluate_Cosmos3.ipynb)

# Your labeled video data

**Labels first:** footage alone cannot produce an accuracy score. You need questions, answer options, and actual ground-truth answers. Confirm data permissions before using a GPU.

Follow the [standalone customer-data recipe](CUSTOM_DATA_RECIPE.md) to map your source files into canonical JSONL and a config JSON. This is a customer-defined evaluation, not an official benchmark or leaderboard submission.


In [ ]:
import json
import os
from pathlib import Path
import sys
from datetime import datetime, timezone
from notebook_ui import invoke, show_customer_results

CLI = [sys.executable, "-u", str(Path.cwd() / "custom_eval.py")]


## 01 / Point to your configuration

Set the absolute path to your external config file. It names your manifest, video directory, model, frame/pixel budget, and generation settings. Paths inside that file resolve relative to the config—not this notebook. Keep inputs fixed when comparing models.


In [ ]:
CONFIG_FILE = Path("/datasets/customer-eval/config-nano.json")
API_NPROC = 1  # 1–4; run one benchmark notebook at a time.

RESULTS_ROOT = Path(os.environ.get("EVAL_RESULTS_DIR", "/results"))
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
WORK = RESULTS_ROOT / "customer-video-mcq" / f"custom-{STAMP}"
LOG = WORK.with_suffix(".log")
if not CONFIG_FILE.is_absolute():
    raise ValueError("Set CONFIG_FILE to your absolute external configuration path.")
print(f"Configuration: {CONFIG_FILE}\nNew result directory: {WORK}")


## 02 / Validate labels and media

This CPU-only step checks every selected question, label, ID, option list, video, and temporal bound before any model request. Missing or ambiguous inputs stop the run; they are not silently skipped. Review the reported count and dataset identity.


In [ ]:
invoke(CLI + ["validate", "--config", str(CONFIG_FILE)], WORK.with_suffix(".validation.log"))


## 03 / Run and score locally

The endpoint must already serve the model named in your configuration. Only the question and all options are sent—not the correct-answer field. Invalid or truncated answers count wrong; transport failures cannot become a completed score. No external judge or manual benchmark command.


In [ ]:
invoke(CLI + ["run", "--config", str(CONFIG_FILE), "--work-dir", str(WORK),
              "--api-nproc", str(API_NPROC)], LOG)


## 04 / Inspect the score and execution evidence

Check complete coverage, category scores, dataset identity, and requested versus observable effective input settings. An unknown effective setting is not a verified setting. Retain the model revision and any budget differences.


In [ ]:
summary = json.loads((WORK / "summary.json").read_text())
show_customer_results(summary, WORK, LOG)


## Compare another model

Deploy or select the second endpoint using the [recipe](CUSTOM_DATA_RECIPE.md#6-inspect-local-scoring-and-compare-models). Change only the model/revision and any required processor profile in a copy of your config. Keep the manifest and videos unchanged; rerun this notebook for a fresh result directory. Require equal dataset identities and coverage before comparing scores. Closing a notebook tab does not stop a running kernel or Brev VM billing.

Share reviewed aggregate scores and code—not labels, videos, raw predictions, credentials, or executed notebooks.
